# Exploracao Compilada dos Sensores

Este notebook consolida os artefatos gerados pelos notebooks individuais:
- `exploracao_sentinel2_long.ipynb`
- `exploracao_landsat89_long.ipynb`
- `exploracao_modis_long.ipynb`

Ele compara estatisticas de bandas, indices e hipoteses para orientar a etapa de modelagem integrada.


## 1) Setup


In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'src').exists() and (cwd / 'notebooks').exists():
        return cwd
    if cwd.name == 'notebooks' and (cwd.parent / 'src').exists():
        return cwd.parent
    return cwd


## 2) Carregamento dos artefatos


In [ ]:
PROJECT_ROOT = resolve_project_root()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'eda_long_format'

SENSORS = {
    'sentinel2': 'Sentinel-2',
    'landsat89': 'Landsat 8/9',
    'modis': 'MODIS',
}

print('Lendo artefatos em:', OUTPUT_DIR)


def load_sensor_artifacts(sensor_key: str):
    overview_path = OUTPUT_DIR / f'{sensor_key}_overview.json'
    band_path = OUTPUT_DIR / f'{sensor_key}_band_summary.csv'
    index_path = OUTPUT_DIR / f'{sensor_key}_index_summary.csv'
    hypothesis_path = OUTPUT_DIR / f'{sensor_key}_hypotheses.txt'
    image_counts_path = OUTPUT_DIR / f'{sensor_key}_image_counts.csv'

    data = {
        'overview': None,
        'band_summary': pd.DataFrame(),
        'index_summary': pd.DataFrame(),
        'hypotheses': [],
        'image_counts': pd.DataFrame(),
    }

    if overview_path.exists():
        data['overview'] = json.loads(overview_path.read_text(encoding='utf-8'))
    if band_path.exists():
        data['band_summary'] = pd.read_csv(band_path)
    if index_path.exists():
        data['index_summary'] = pd.read_csv(index_path)
    if hypothesis_path.exists():
        data['hypotheses'] = [
            line.strip() for line in hypothesis_path.read_text(encoding='utf-8').splitlines() if line.strip()
        ]
    if image_counts_path.exists():
        data['image_counts'] = pd.read_csv(image_counts_path)

    return data


artifacts = {k: load_sensor_artifacts(k) for k in SENSORS}

for sensor_key, artifact in artifacts.items():
    ok = artifact['overview'] is not None and not artifact['band_summary'].empty
    status = 'OK' if ok else 'INCOMPLETO'
    print(f'- {SENSORS[sensor_key]}: {status}')


## 3) Resumo geral por sensor


In [ ]:
overview_rows = []
for sensor_key, artifact in artifacts.items():
    overview = artifact['overview']
    if overview is None:
        continue
    row = {'sensor_key': sensor_key, 'sensor_name': SENSORS[sensor_key], **overview}
    overview_rows.append(row)

overview_df = pd.DataFrame(overview_rows)
display(overview_df)

if not overview_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    sns.barplot(data=overview_df, x='sensor_name', y='n_rows_total', ax=axes[0])
    axes[0].set_title('Linhas totais')
    axes[0].tick_params(axis='x', rotation=20)

    sns.barplot(data=overview_df, x='sensor_name', y='n_bands', ax=axes[1])
    axes[1].set_title('Numero de bandas')
    axes[1].tick_params(axis='x', rotation=20)

    if 'n_images' in overview_df.columns:
        sns.barplot(data=overview_df, x='sensor_name', y='n_images', ax=axes[2])
        axes[2].set_title('Numero de imagens')
        axes[2].tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()


## 4) Comparativo de estatisticas por banda


In [ ]:
band_frames = []
for sensor_key, artifact in artifacts.items():
    df = artifact['band_summary']
    if df.empty:
        continue
    df = df.copy()
    df['sensor_key'] = sensor_key
    df['sensor_name'] = SENSORS[sensor_key]
    band_frames.append(df)

if band_frames:
    band_all = pd.concat(band_frames, ignore_index=True)
    display(band_all.head())

    common_bands = (
        band_all.groupby('band')['sensor_key'].nunique()
        .pipe(lambda s: s[s == len(band_all['sensor_key'].unique())])
        .index
        .tolist()
    )

    if common_bands:
        mean_pivot = (
            band_all[band_all['band'].isin(common_bands)]
            .pivot_table(index='sensor_name', columns='band', values='mean')
            .sort_index(axis=1)
        )

        std_pivot = (
            band_all[band_all['band'].isin(common_bands)]
            .pivot_table(index='sensor_name', columns='band', values='std')
            .sort_index(axis=1)
        )

        fig, axes = plt.subplots(2, 1, figsize=(14, 8))
        sns.heatmap(mean_pivot, cmap='viridis', ax=axes[0])
        axes[0].set_title('Media por banda (bandas em comum)')

        sns.heatmap(std_pivot, cmap='magma', ax=axes[1])
        axes[1].set_title('Desvio padrao por banda (bandas em comum)')

        plt.tight_layout()
        plt.show()
    else:
        print('Nao ha intersecao completa de bandas entre todos os sensores.')

    missing_view = band_all.groupby('sensor_name')['missing_pct'].mean().sort_values()
    display(missing_view.to_frame('missing_pct_medio'))
else:
    band_all = pd.DataFrame()
    print('Nenhum band_summary encontrado.')


## 5) Comparativo de indices espectrais


In [ ]:
index_frames = []
for sensor_key, artifact in artifacts.items():
    idx_df = artifact['index_summary']
    if idx_df.empty:
        continue
    idx_df = idx_df.copy()
    idx_df['sensor_key'] = sensor_key
    idx_df['sensor_name'] = SENSORS[sensor_key]
    index_frames.append(idx_df)

if index_frames:
    index_all = pd.concat(index_frames, ignore_index=True)
    display(index_all[['sensor_name', 'index_name', 'mean', 'std', 'min', 'max']])

    plt.figure(figsize=(10, 5))
    sns.barplot(data=index_all, x='index_name', y='mean', hue='sensor_name')
    plt.title('Media dos indices por sensor')
    plt.tight_layout()
    plt.show()
else:
    index_all = pd.DataFrame()
    print('Nenhum indice foi salvo nos artefatos.')


## 6) Hipoteses consolidadas


In [ ]:
hyp_rows = []
for sensor_key, artifact in artifacts.items():
    for idx, text in enumerate(artifact['hypotheses'], start=1):
        hyp_rows.append({
            'sensor_name': SENSORS[sensor_key],
            'ordem': idx,
            'hipotese': text,
        })

hyp_df = pd.DataFrame(hyp_rows)
display(hyp_df)

integrated_hypotheses = []

if not overview_df.empty and 'n_rows_total' in overview_df.columns:
    largest = overview_df.sort_values('n_rows_total', ascending=False).iloc[0]['sensor_name']
    integrated_hypotheses.append(
        f'O sensor com maior volume de pixels e {largest}; testar estrategias de balanceamento na etapa integrada.'
    )

if not band_all.empty:
    high_missing_sensor = band_all.groupby('sensor_name')['missing_pct'].mean().sort_values(ascending=False).index[0]
    integrated_hypotheses.append(
        f'{high_missing_sensor} apresentou maior taxa media de faltantes nas bandas; avaliar imputacao/mascara especifica por sensor.'
    )

if not index_all.empty and 'NDVI' in index_all['index_name'].values:
    ndvi_spread = (
        index_all[index_all['index_name'] == 'NDVI']
        .set_index('sensor_name')['mean']
        .sort_values()
    )
    if len(ndvi_spread) >= 2:
        integrated_hypotheses.append(
            'As medias de NDVI variam entre sensores; testar normalizacao por sensor antes de treinar modelos unificados.'
        )

if not integrated_hypotheses:
    integrated_hypotheses.append('Rodar todos os notebooks individuais para gerar evidencias comparativas mais fortes.')

print('Hipoteses integradas')
for i, h in enumerate(integrated_hypotheses, start=1):
    print(f'{i}. {h}')


## 7) Exportacao do compilado


In [ ]:
compile_overview_path = OUTPUT_DIR / 'compilado_overview.csv'
compile_hyp_path = OUTPUT_DIR / 'compilado_hipoteses.txt'

if not overview_df.empty:
    overview_df.to_csv(compile_overview_path, index=False)

compile_hyp_path.write_text('\n'.join(integrated_hypotheses), encoding='utf-8')

print('Arquivos salvos:')
if compile_overview_path.exists():
    print('-', compile_overview_path)
print('-', compile_hyp_path)
